# Week 1 Homework — Grounded Reply Assistant (Mini-Project)

**One notebook that ties all of Week 1 together.**

You will build a small assistant that:
1. **Retrieves** facts from the company handbook using embeddings (RAG)
2. **Decides** which tool to call: `search_handbook` / `buy_item` / `return_item`
3. **Answers** in a clean structured format (Pydantic) with citations
4. **Proves** it works with a mini eval test

> **Backend:** Groq API `gpt-oss-20b:groq` + `nomic-embed-text-v1.5`
> **Structured output:** `instructor` patches the client so `response_model=GroundedReply` works on Groq

## Step 0 — Setup & Imports (Dual-Model Architecture)

We use a **CWD-relative path** and patch the Groq client with **`instructor`** so we can use `response_model=GroundedReply` in chat completions.

**Architecture Upgrade (Fallback Mechanism):**
This setup defines **two** LLM clients:
1. **Primary Model**: HuggingFace Router (`gpt-oss-20b:groq`) for superior reasoning on multi-part queries.
2. **Fallback Model**: Native Groq API (`llama-3.1-8b-instant`).

If the Primary Model fails (e.g., due to a 402/403 credits error), the custom `execute_with_fallback` function will seamlessly intercept the error and execute the request on the Fallback Model, ensuring the application never crashes.

In [1]:
pip install einops

Note: you may need to restart the kernel to use updated packages.


In [8]:
# ---------------------------------------------------------------------------
# Step 0 — Credentials & Clients
# ---------------------------------------------------------------------------
import sys, pathlib, json, uuid, os, re
import chromadb
import instructor
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import openai

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

# Ensure masar_utils is in path
import pathlib, sys
_REPO_ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "Homework Week" / "Homework Week 1" / "shared"))
from masar_utils import pretty, count_tokens, estimate_cost, get_client

# -- 1. Primary Client (HuggingFace Router) --
_HF_BASE_URL = os.environ.get("HF_BASE_URL", "https://router.huggingface.co/v1")
_HF_TOKEN    = os.environ.get("HF_TOKEN", "")
PRIMARY_MODEL = os.environ.get("HF_MODEL", "openai/gpt-oss-20b:groq")

if not _HF_TOKEN:
    print("Warning: HF_TOKEN not found, falling back directly to Groq")
    _primary_raw_client = None
    primary_client = None
else:
    _primary_raw_client = OpenAI(base_url=_HF_BASE_URL, api_key=_HF_TOKEN)
    primary_client = instructor.from_openai(_primary_raw_client, mode=instructor.Mode.MD_JSON)

# -- 2. Fallback Client (Groq directly) --
os.environ.setdefault("OPENAI_BASE_URL", "https://api.groq.com/openai/v1")
FALLBACK_MODEL = "llama-3.1-8b-instant"
_fallback_raw_client = get_client()
fallback_client = instructor.from_openai(_fallback_raw_client, mode=instructor.Mode.MD_JSON)

MODEL = PRIMARY_MODEL
print(f"Primary model : {PRIMARY_MODEL}")
print(f"Fallback model: {FALLBACK_MODEL}")

# -- 3. Embeddings --
os.environ.setdefault("OPENAI_EMBEDDING_MODEL", "nomic-embed-text-v1.5")
print("Loading embedding model...")
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
print("Clients and embedding model ready!")

# Global variable to track which model was actually used
_last_model_used = None

def get_last_model_used():
    global _last_model_used
    return _last_model_used

# Helper for fallback execution
def execute_with_fallback(*args, **kwargs):
    global _last_model_used
    """Executes a chat completion call with the primary client, falling back if it fails."""
    client_inst_from_kwargs = kwargs.pop('client_instance', None)
    primary_inst = client_inst_from_kwargs if client_inst_from_kwargs is not None else _primary_raw_client
    
    try:
        if _primary_raw_client is None:
            raise openai.APIStatusError("No HF token", response=None, body=None) # type: ignore
        
        kwargs['model'] = PRIMARY_MODEL
        print(f"  [🤖 Model Executing] {PRIMARY_MODEL}")
        _last_model_used = PRIMARY_MODEL
        return primary_inst.chat.completions.create(*args, **kwargs)
    except Exception as e:
        print(f"  [⚠️ Fallback Triggered] Primary failed: {type(e).__name__} - {e}")
        if hasattr(primary_inst, 'mode'):
            fallback_inst = fallback_client
        else:
            fallback_inst = _fallback_raw_client
            
        kwargs['model'] = FALLBACK_MODEL
        print(f"  [🤖 Model Executing] {FALLBACK_MODEL}")
        _last_model_used = FALLBACK_MODEL
        return fallback_inst.chat.completions.create(*args, **kwargs)


Primary model : openai/gpt-oss-20b:groq
Fallback model: llama-3.1-8b-instant
Loading embedding model...


<All keys matched successfully>


Clients and embedding model ready!


---
# Task 1 · Retrieve (Embeddings RAG)

> **Reference:** `05_rag.ipynb` – Sections 2-7

Make the handbook searchable using ChromaDB and nomic-embed-text-v1.5.

### 1.1 — Load the handbook corpus

In [7]:
DOC_PATH = pathlib.Path.cwd() / "data/company_handbook.md"
raw_text = DOC_PATH.read_text(encoding="utf-8")
print(f"Loaded {len(raw_text)} characters from: {DOC_PATH}")
print(raw_text[:500], "...")

Loaded 2162 characters from: c:\Users\HUAWEI\Downloads\‏‏Masar-ai agent\week1\homework\data\company_handbook.md
# Acme Cloud, Company Handbook & Product FAQ (sample RAG corpus)

This is synthetic sample data for the RAG sessions and the email/article projects.

## About Acme Cloud
Acme Cloud is a (fictional) platform that provides managed Postgres, object storage,
and serverless functions. Founded in 2019, headquartered in Riyadh, Saudi Arabia, with a
remote-first team across the MENA region.

## Plans & Pricing
- **Starter**, $0/month. 1 project, 1 GB Postgres, 5 GB storage, community support.
- **Pro**, ...


### 1.2 — Chunk the text (Header + Bullet Chunking)

**Architecture Upgrade:** Previously, chunking was done purely at the `##` header level, resulting in massive text blocks that diluted the embedding vectors for short queries (like 'reset password').

We have upgraded `chunk_by_headers()` to detect bulleted lists and split them into individual `[Header] + [Bullet]` chunks. This drastically improves semantic density and RAG precision.

In [4]:
import re

def chunk_by_headers(text):
    """
    Split markdown text into chunks based on level-2 headers (##).
    """
    # تقسيم النص عند كل عنوان ##
    sections = re.split(r'(?=^## )', text, flags=re.MULTILINE)

    chunks = []

    # أول جزء قبل أول عنوان (العنوان الرئيسي والوصف)
    if sections[0].strip():
        chunks.append(sections[0].strip())

    # بقية الأقسام
    for section in sections[1:]:
        chunks.append(section.strip())

    return chunks


chunks = chunk_by_headers(raw_text)

print(f"Created {len(chunks)} chunks")

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(c)} chars) ---")
    print(c[:200], "...")

Created 8 chunks

--- Chunk 0 (148 chars) ---
# Acme Cloud, Company Handbook & Product FAQ (sample RAG corpus)

This is synthetic sample data for the RAG sessions and the email/article projects. ...

--- Chunk 1 (235 chars) ---
## About Acme Cloud
Acme Cloud is a (fictional) platform that provides managed Postgres, object storage,
and serverless functions. Founded in 2019, headquartered in Riyadh, Saudi Arabia, with a
remote ...

--- Chunk 2 (405 chars) ---
## Plans & Pricing
- **Starter**, $0/month. 1 project, 1 GB Postgres, 5 GB storage, community support.
- **Pro**, $29/month. 10 projects, 50 GB Postgres, 250 GB storage, email support, daily backups.
 ...

--- Chunk 3 (264 chars) ---
## Support & SLA
- Pro plan: email support, first response within 24 business hours.
- Scale plan: priority support, first response within 4 business hours, 99.95% uptime SLA.
- Status page: status.ac ...

--- Chunk 4 (213 chars) ---
## Backups & Recovery
- Pro: automatic daily backups retained for 7 da

### 1.3 — ChromaDB Vector Database

Production-ready vector DB with native cosine similarity.

In [5]:
chroma_client = chromadb.Client()
vector_db = chroma_client.get_or_create_collection(
    name="handbook",
    metadata={"hnsw:space": "cosine"},
)
print("ChromaDB in-memory client created")
print(f"Collection: '{vector_db.name}' (cosine similarity)")

ChromaDB in-memory client created
Collection: 'handbook' (cosine similarity)


### 1.4 — Embed all chunks & build the index

We use `_raw_client` (the unpatched client) for embeddings, because instructor
only patches chat completions — embeddings are unaffected.

> **Reference:** `05_rag.ipynb` – Sections 4 & 6

In [6]:
def embed(texts):
    """Embed a list of strings using SentenceTransformer."""
    if isinstance(texts, str):
        texts = [texts]

    return embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).tolist()

print("Embedding all chunks via SentenceTransformer...")
chunk_vecs = embed(chunks)

vector_db.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_vecs,
)

print(f"Indexed {len(chunks)} chunks into ChromaDB")
print(f"Collection count: {vector_db.count()}")
print(f"Vector dimensions: {len(chunk_vecs[0])}")

Embedding all chunks via SentenceTransformer...
Indexed 8 chunks into ChromaDB
Collection count: 8
Vector dimensions: 768


In [7]:
result = vector_db.get(ids=["chunk-0"], include=["documents", "embeddings"])

print("ID:", result["ids"][0])
print("\nDocument:\n")
print(result["documents"][0])
print("\nEmbedding length:", len(result["embeddings"][0]))
print("\nFirst 10 values:")
print(result["embeddings"][0][:10])

ID: chunk-0

Document:

# Acme Cloud, Company Handbook & Product FAQ (sample RAG corpus)

This is synthetic sample data for the RAG sessions and the email/article projects.

Embedding length: 768

First 10 values:
[ 0.02265126  0.02823102 -0.19923559 -0.077705    0.03653704 -0.05240844
  0.02660957 -0.00214236 -0.02448259 -0.02910927]


### 1.5 — Retrieve top-k chunks (with similarity threshold)

Returns empty list when best score < threshold — forces refusal on off-topic questions.

In [8]:
SIMILARITY_THRESHOLD = 0.55

def retrieve(question, k=3):
    """
    Embed the question, query ChromaDB, return top-k chunks.
    Returns EMPTY list when best score < SIMILARITY_THRESHOLD.
    ChromaDB cosine distance: 0=identical. sim = 1 - distance.
    """
    q_emb = embed([question])[0]
    results = vector_db.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "distances"],
    )
    docs  = results["documents"][0]
    dists = results["distances"][0]
    hits  = [(doc, 1.0 - dist) for doc, dist in zip(docs, dists)]
    if not hits or hits[0][1] < SIMILARITY_THRESHOLD:
        return []
    return hits

print("=== Test: handbook question ===")
hits = retrieve("How long are Pro backups kept?")
for i, (doc, score) in enumerate(hits, 1):
    print(f"  #{i} (score={score:.3f}): {doc[:100]}...")
print("\n=== Test: off-topic question ===")
hits = retrieve("What is the GDP of Saudi Arabia?")
print(f"  Hits returned: {len(hits)} (should be 0 or very low relevance)")

=== Test: handbook question ===
  #1 (score=0.755): ## Backups & Recovery
- Pro: automatic daily backups retained for 7 days.
- Scale: PITR with 1-secon...
  #2 (score=0.570): ## Plans & Pricing
- **Starter**, $0/month. 1 project, 1 GB Postgres, 5 GB storage, community suppor...
  #3 (score=0.555): ## Common Support Answers
- **Reset password:** dashboard → Account → Security → "Send reset link".
...

=== Test: off-topic question ===
  Hits returned: 0 (should be 0 or very low relevance)


---
# Task 2 · Decide (Function / Tool Calling)

> **Reference:** `02_function_calling.ipynb` – Sections 3-8

Three tools: `search_handbook`, `buy_item`, `return_item`.

### 2.1 — Dummy store data

In [9]:
CATALOG = {
    "sku-tee": {"name": "Acme T-Shirt", "price": 20.0, "stock": 10},
    "sku-mug": {"name": "Acme Mug",     "price": 12.0, "stock": 5},
    "sku-cap": {"name": "Acme Cap",     "price": 15.0, "stock": 0},
}
ORDERS = {}
print("Catalog:")
for sku, info in CATALOG.items():
    print(f"  {sku}: {info['name']} — ${info['price']:.2f} (stock: {info['stock']})")

Catalog:
  sku-tee: Acme T-Shirt — $20.00 (stock: 10)
  sku-mug: Acme Mug — $12.00 (stock: 5)
  sku-cap: Acme Cap — $15.00 (stock: 0)


### 2.2 — Tool function implementations

In [10]:
def search_handbook(query: str) -> dict:
    """Look up Acme Cloud policy/pricing/support facts via RAG."""
    hits = retrieve(query)
    if not hits:
        return {"context": "", "found": False,
                "message": "No relevant handbook content found for this question."}
    context = "\n\n".join(f"[{i+1}] {doc}" for i, (doc, _) in enumerate(hits))
    return {"context": context, "found": True}


def buy_item(item_id: str, quantity: int) -> dict:
    if quantity < 1:
        return {"error": "InvalidQuantity", "message": "Quantity must be at least 1."}
    """Create an order. Returns error if item unknown or out of stock."""
    if item_id not in CATALOG:
        return {"error": "ItemNotFound",
                "message": f"'{item_id}' not in catalog. Available: {list(CATALOG.keys())}"}
    item = CATALOG[item_id]
    if item["stock"] < quantity:
        return {"error": "OutOfStock",
                "message": f"Only {item['stock']} unit(s) of '{item['name']}' in stock (requested {quantity})."}
    item["stock"] -= quantity
    order_id = "ord-" + uuid.uuid4().hex[:8]
    total    = round(item["price"] * quantity, 2)
    ORDERS[order_id] = {"item_id": item_id, "item_name": item["name"],
                        "quantity": quantity, "total": total, "status": "confirmed"}
    return {"order_id": order_id, "item": item["name"],
            "quantity": quantity, "total": total, "status": "confirmed"}


def return_item(order_id: str) -> dict:
    """Mark an existing order as returned."""
    if order_id not in ORDERS:
        return {"error": "OrderNotFound", "message": f"Order '{order_id}' does not exist."}
    order = ORDERS[order_id]
    if order["status"] == "returned":
        return {"error": "AlreadyReturned", "message": f"Order '{order_id}' was already returned."}
    CATALOG[order["item_id"]]["stock"] += order["quantity"]
    order["status"] = "returned"
    return {"order_id": order_id, "status": "returned", "refunded": order["total"],
            "message": f"Order '{order_id}' returned, ${order['total']:.2f} refunded."}


print("Tool functions defined: search_handbook, buy_item, return_item")

Tool functions defined: search_handbook, buy_item, return_item


### 2.3 — Quick test of tool functions

In [11]:
result = search_handbook("What are the Pro plan backups?")
print("search_handbook ->", "found" if result["found"] else "not found")
result = buy_item("sku-mug", 1)
print("buy_item(sku-mug, 1) ->", result)
test_order_id = result.get("order_id", "")
result = buy_item("sku-cap", 1)
print("buy_item(sku-cap, 1) ->", result)
result = return_item(test_order_id)
print(f"return_item({test_order_id}) ->", result)
result = return_item("ord-fake")
print("return_item(ord-fake) ->", result)

search_handbook -> found
buy_item(sku-mug, 1) -> {'order_id': 'ord-d8d35bbd', 'item': 'Acme Mug', 'quantity': 1, 'total': 12.0, 'status': 'confirmed'}
buy_item(sku-cap, 1) -> {'error': 'OutOfStock', 'message': "Only 0 unit(s) of 'Acme Cap' in stock (requested 1)."}
return_item(ord-d8d35bbd) -> {'order_id': 'ord-d8d35bbd', 'status': 'returned', 'refunded': 12.0, 'message': "Order 'ord-d8d35bbd' returned, $12.00 refunded."}
return_item(ord-fake) -> {'error': 'OrderNotFound', 'message': "Order 'ord-fake' does not exist."}


### 2.4 — Tool schemas (OpenAI-compatible format)

Groq uses the standard OpenAI tool-calling format. We omit `strict: true`.

In [12]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_handbook",
            "description": (
                "Look up Acme Cloud policy/pricing/support facts. "
                "Use when the user asks about the company, plans, SLA, backups, refunds, or support."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Natural-language question to look up in the handbook."
                    }
                },
                "required": ["query"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buy_item",
            "description": "Purchase an item from the Acme store. Use when the user wants to buy something.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_id": {
                        "type": "string",
                        "description": "Catalog SKU. Must be exactly one of: sku-tee, sku-mug, sku-cap."
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "Number of units (>= 1)."
                    }
                },
                "required": ["item_id", "quantity"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "return_item",
            "description": "Return a previously placed order. Use when the user wants to return or cancel an order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID to return."
                    }
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

print("Tool schemas defined:", [t["function"]["name"] for t in TOOLS])

Tool schemas defined: ['search_handbook', 'buy_item', 'return_item']


### 2.5 — `safe_call` — error-safe tool runner

> **Reference:** `02_function_calling.ipynb` – Section 6

In [13]:
def safe_call(func, args):
    """Run a tool returning JSON string. On failure return structured error."""
    try:
        return json.dumps(func(**args))
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})

TOOL_IMPLS = {
    "search_handbook": search_handbook,
    "buy_item":        buy_item,
    "return_item":     return_item,
}

print("safe_call with bad args:", safe_call(buy_item, {"item_id": "sku-fake", "quantity": 1}))

safe_call with bad args: {"error": "ItemNotFound", "message": "'sku-fake' not in catalog. Available: ['sku-tee', 'sku-mug', 'sku-cap']"}


### 2.6 — `run_agent()` — the multi-round tool-call loop

> **Reference:** `02_function_calling.ipynb` — Section 7 (reused verbatim, adapted to this project)

The loop:
1. Call the model with `tools=TOOLS, tool_choice="auto"`
2. If the response has `tool_calls` → run each via `safe_call()` → append `assistant` + `tool` messages
3. Repeat until `finish_reason == "stop"` (no more tool calls) or `max_rounds` is reached
4. Return the final text content

**Parallel tool calls** are handled: a single assistant turn may request multiple calls;
we execute every one and send back one `tool` message per `tool_call_id`.


In [14]:
def run_agent(
    user_msg: str,
    tools: list = TOOLS,
    tool_impls: dict = TOOL_IMPLS,
    system: str = "You are a helpful assistant. Use tools when useful.",
    max_rounds: int = 5,
    verbose: bool = True,
) -> str:
    """
    Multi-round tool-call loop.

    Parameters
    ----------
    user_msg    : str   The user message to process.
    tools       : list  Tool JSON schemas (defaults to TOOLS).
    tool_impls  : dict  {name -> callable} map (defaults to TOOL_IMPLS).
    system      : str   System prompt.
    max_rounds  : int   Safety cap on the number of model calls.
    verbose     : bool  Print each round and tool call.

    Returns
    -------
    str  The model's final text answer.
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_msg},
    ]

    for round_i in range(1, max_rounds + 1):
        resp  = _raw_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0,
            parallel_tool_calls=False,
        )
        msg   = resp.choices[0].message
        calls = msg.tool_calls or []

        if resp.choices[0].finish_reason == "stop":
            if verbose:
                print(f"[round {round_i}] final answer")
            return msg.content

        if verbose:
            print(f"[round {round_i}] model requested {len(calls)} tool call(s): "
                  + ", ".join(c.function.name for c in calls))

        messages.append(msg.model_dump(exclude_none=True))
        for c in calls:
            name    = c.function.name
            args    = json.loads(c.function.arguments or "{}")
            impl    = tool_impls.get(name)
            content = json.dumps({"error": "UnknownTool", "message": name}) if impl is None else safe_call(impl, args)
            if verbose:
                print(f"    {name}({args}) -> {content[:200]}")
            messages.append({"role": "tool", "tool_call_id": c.id, "content": content})

    return "(stopped: hit max_rounds without a final answer)"


# ---------------------------------------------------------------------------
# Task 3 · Answer (Structured + Grounded)
# ---------------------------------------------------------------------------

---
# Task 3 · Answer (Structured + Grounded)

> **Reference:** `03_structured_output.ipynb` – Section 4

**How `instructor` replaces `client.beta.chat.completions.parse`:**

| Original (OpenAI only) | This notebook (Groq + instructor) |
|------------------------|------------------------------------|
| `client.beta.chat.completions.parse(response_format=GroundedReply)` | `client.chat.completions.create(response_model=GroundedReply)` |
| Returns `.choices[0].message.parsed` | Returns the Pydantic object directly |
| Only works with OpenAI | Works with any OpenAI-compatible API |

### 3.1 — Pydantic `GroundedReply` model

In [15]:
class GroundedReply(BaseModel):
    """Structured response model for the Grounded Reply Assistant."""
    answer:     str       = Field(description="The assistant's answer to the user.")
    sources:    list[str] = Field(description="Titles or identifiers of the handbook sections used to answer the question.")
    confidence: float     = Field(description="Confidence score 0.0-1.0.", ge=0.0, le=1.0)
    answered:   bool      = Field(description="True ONLY if the user's request was fulfilled using the handbook context or store tools; False for greetings, off-topic, or if no grounded answer was provided.")

print("GroundedReply model defined")
print(json.dumps(GroundedReply.model_json_schema(), indent=2))

GroundedReply model defined
{
  "description": "Structured response model for the Grounded Reply Assistant.",
  "properties": {
    "answer": {
      "description": "The assistant's answer to the user.",
      "title": "Answer",
      "type": "string"
    },
    "sources": {
      "description": "Titles or identifiers of the handbook sections used to answer the question.",
      "items": {
        "type": "string"
      },
      "title": "Sources",
      "type": "array"
    },
    "confidence": {
      "description": "Confidence score 0.0-1.0.",
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidence",
      "type": "number"
    },
    "answered": {
      "description": "True ONLY if the user's request was fulfilled using the handbook context or store tools; False for greetings, off-topic, or if no grounded answer was provided.",
      "title": "Answered",
      "type": "boolean"
    }
  },
  "required": [
    "answer",
    "sources",
    "confidence",
    "answered"
  ]

### 3.2 — System prompt

Prompt anatomy from **notebook 04**: Role, Context, Task, Constraints, Format.

In [16]:
TOOL_SYSTEM_PROMPT = """\
Role: You are a precise, grounded support assistant for Acme Cloud.

Task:
1. If the user asks about Acme Cloud topics (company info, pricing, plans, SLA, backups,
   refunds, security, regions, connection limits, data export, support, passwords),
   you MUST call search_handbook — even for short phrases like "reset password".
2. If the user wants to buy or return an item, use the appropriate store tool.
3. For anything unrelated to Acme Cloud (world events, geography, general knowledge),
   do NOT call any tool — just reply in plain text that you can only help with Acme topics.

Constraints:
- ONLY use the tools provided: search_handbook, buy_item, return_item. Never invent or call other tools.
- Never guess or hallucinate facts.
- Use EXACTLY the quantity the user specified for each item. Never assume, round, or change a quantity.
- ONLY call return_item if the user explicitly asks to return, cancel, or refund an order. NEVER call return_item right after a purchase unless the user separately and explicitly requested a return.
- Only call each tool for actions the user actually asked for. Do not perform extra, unrequested actions."""

STRUCTURED_SYSTEM_PROMPT = """\
Role: You are a precise, grounded support assistant for Acme Cloud formatting the final response.

Constraints for GroundedReply:
- The user's message may contain MULTIPLE requests (e.g. a question + a purchase).
  You MUST address every part of it in `answer`. Never drop a part just because
  it wasn't the last thing discussed.
- For handbook questions, answer ONLY from the raw tool context provided.
- For store actions, confirm the result (order id, total, or error) from the raw tool context.
- Populate `sources` ONLY with handbook section titles actually used (from search_handbook results).
  If no handbook lookup contributed, sources = [].
- Never invent handbook section titles; use ONLY titles that appear exactly in the retrieved context.
- If a tool never returned a result for part of the request, say so — don't guess."""

print(f"System prompts defined")

System prompts defined


### 3.3 — `grounded_assistant()` — the full pipeline

Combines the **agent loop** (Task 2) + **instructor structured parse** (Task 3).

**Key design:**
- `_raw_client` handles the tool-calling loop (instructor not involved)
- `client` (instructor-patched) handles the final structured parse
- `response_model=GroundedReply` returns the Pydantic object directly

> **Reference:** `02_function_calling.ipynb` – Section 7, `03_structured_output.ipynb` – Section 4

In [17]:
def grounded_assistant(
    user_msg: str,
    max_rounds: int = 5,
    verbose: bool = True,
) -> tuple["GroundedReply", list[str]]:
    """
    Run the full Grounded Reply pipeline for one user message.

    Returns
    -------
    reply        : GroundedReply  - the parsed, structured answer
    tools_called : list[str]      - names of every tool that was invoked
    """
    messages = [
        {"role": "system", "content": TOOL_SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    tools_called: list[str] = []
    tool_results_log: list[str] = []

    # Agent loop (uses _raw_client so instructor does not intercept)
    for round_i in range(1, max_rounds + 1):
        resp = execute_with_fallback(
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
            parallel_tool_calls=False,
            client_instance=_primary_raw_client
        )
        msg   = resp.choices[0].message
        calls = msg.tool_calls or []

        if resp.choices[0].finish_reason == "stop":
            if verbose:
                print(f"  [round {round_i}] no tool calls -> structured parse")
            messages.append({"role": "assistant", "content": msg.content or ""})
            break

        if verbose:
            print(f"  [round {round_i}] {len(calls)} tool call(s): "
                  + ", ".join(c.function.name for c in calls))
        messages.append(msg.model_dump(exclude_none=True))

        for tc in calls:
            name  = tc.function.name
            args  = json.loads(tc.function.arguments or "{}")
            impl  = TOOL_IMPLS.get(name)
            if impl is None:
                content = json.dumps({"error": "UnknownTool", "message": name})
            else:
                content = safe_call(impl, args)
                tools_called.append(name)
                tool_results_log.append(f"[{name}] {content}")
            if verbose:
                print(f"    {name}({args}) -> {content[:200]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
    else:
        if verbose:
            print(f"  [warning] hit max_rounds={max_rounds}")

    context_block = "\n\n".join(tool_results_log) if tool_results_log else "(no tools were called)"

    structured_messages = [
        {"role": "system", "content": STRUCTURED_SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
        {"role": "system", "content": f"Raw tool results (ground truth — cover ALL of these):\n{context_block}"},
    ]

    reply: GroundedReply = execute_with_fallback(
        messages=structured_messages,
        response_model=GroundedReply,
        temperature=0,
        client_instance=primary_client
    )

    if verbose:
        prompt_text = "".join(m.get("content", "") for m in structured_messages if isinstance(m, dict) and "content" in m)
        comp_text   = reply.answer if hasattr(reply, "answer") else ""
        p_toks = count_tokens(prompt_text)
        c_toks = count_tokens(comp_text)
        cost   = estimate_cost(p_toks, c_toks)
        print(f"  [Cost Estimate] ~{p_toks} prompt tokens, ~{c_toks} completion tokens => ${cost:.6f}")

    return reply, tools_called

### 3.4 — Demo: test the full pipeline

In [18]:
CATALOG["sku-tee"]["stock"] = 10
CATALOG["sku-mug"]["stock"] = 5
CATALOG["sku-cap"]["stock"] = 0
ORDERS.clear()

demo_questions = [
 
    "What is the GDP of Saudi Arabia?is there pro plan i want two t shirt"
  
]
for q in demo_questions:
    print("\n" + "=" * 70)
    print(f"USER: {q}")
    reply, tools = grounded_assistant(q)
    pretty(reply.model_dump(), title="GroundedReply")
    print(f"Tools called: {tools}")


USER: What is the GDP of Saudi Arabia?is there pro plan i want two t shirt
  [round 1] 1 tool call(s): search_handbook
    search_handbook({'query': 'pro plan'}) -> {"context": "[1] ## Plans & Pricing\n- **Starter**, $0/month. 1 project, 1 GB Postgres, 5 GB storage, community support.\n- **Pro**, $29/month. 10 projects, 50 GB Postgres, 250 GB storage, email suppo
  [round 2] 1 tool call(s): buy_item
    buy_item({'item_id': 'sku-tee', 'quantity': 2}) -> {"order_id": "ord-3e41e9a4", "item": "Acme T-Shirt", "quantity": 2, "total": 40.0, "status": "confirmed"}
  [round 3] no tool calls -> structured parse
  [Cost Estimate] ~849 prompt tokens, ~71 completion tokens => $0.000170
GroundedReply
{
  "answer": "I’m sorry, I don’t have information on the GDP of Saudi Arabia. The Pro plan is available for $29/month and includes 10 projects, 50 GB Postgres, 250 GB storage, email support, and daily backups. Your order for two Acme T-Shirts has been confirmed: order ID ord-3e41e9a4, total $40.00.",

In [ ]:
CATALOG["sku-tee"]["stock"] = 10
CATALOG["sku-mug"]["stock"] = 5
CATALOG["sku-cap"]["stock"] = 0
ORDERS.clear()

demo_questions = [
    "Hi there!",
    "How long are Pro backups kept?",
    "What is the Scale plan SLA?",
    "What is the GDP of Saudi Arabia?",
    "I'd like to buy 2 mugs.",
    "I want to buy the cap.",
]

for q in demo_questions:
    print("\n" + "=" * 70)
    print(f"USER: {q}")
    reply, tools = grounded_assistant(q)
    pretty(reply.model_dump(), title="GroundedReply")
    print(f"Tools called: {tools}")


USER: Hi there!
  [round 1] no tool calls -> structured parse
  [Cost Estimate] ~570 prompt tokens, ~8 completion tokens => $0.000090
GroundedReply
{
  "answer": "Hello! How can I assist you today?",
  "sources": [],
  "confidence": 0.0,
  "answered": false
}
Tools called: []

USER: How long are Pro backups kept?
  [round 1] 1 tool call(s): search_handbook
    search_handbook({'query': 'Pro backups kept'}) -> {"context": "[1] ## Backups & Recovery\n- Pro: automatic daily backups retained for 7 days.\n- Scale: PITR with 1-second granularity, 35-day retention.\n- Restores can be triggered from the dashboard 
  [round 2] no tool calls -> structured parse
  [Cost Estimate] ~848 prompt tokens, ~9 completion tokens => $0.000133
GroundedReply
{
  "answer": "Pro backups are retained for 7 days.",
  "sources": [
    "Backups & Recovery"
  ],
  "confidence": 1.0,
  "answered": true
}
Tools called: ['search_handbook']

USER: What is the Scale plan SLA?
  [Fallback Triggered] Primary failed: APIS

API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~1152 prompt tokens, ~9 completion tokens => $0.000178
GroundedReply
{
  "answer": "The Scale plan has a 99.95% uptime SLA.",
  "sources": [
    "## Support & SLA"
  ],
  "confidence": 1.0,
  "answered": true
}
Tools called: ['search_handbook']

USER: What is the GDP of Saudi Arabia?
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] no tool calls -> structured parse


API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~916 prompt tokens, ~25 completion tokens => $0.000152
GroundedReply
{
  "answer": "The GDP of Saudi Arabia is approximately $1.73 trillion (nominal) and $1.38 trillion (PPP) as of 2022.",
  "sources": [
    "Economy of Saudi Arabia"
  ],
  "confidence": 0.9,
  "answered": true
}
Tools called: []

USER: I'd like to buy 2 mugs.
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] 1 tool call(s): buy_item
    buy_item({'item_id': 'sku-mug', 'quantity': 2}) -> {"order_id": "ord-421

API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~936 prompt tokens, ~21 completion tokens => $0.000153
GroundedReply
{
  "answer": "Your order for 2 Acme Mugs has been confirmed. Order ID: ord-421b84f9. Total: $24.00.",
  "sources": [],
  "confidence": 1.0,
  "answered": true
}
Tools called: ['buy_item']

USER: I want to buy the cap.
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] 1 tool call(s): buy_item
    buy_item({'item_id': 'sku-cap', 'quantity': 1}) -> {"error": "OutOfStock", "message": "Only 0 unit(s) of 'Acme Ca

API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~933 prompt tokens, ~29 completion tokens => $0.000157
GroundedReply
{
  "answer": "Unfortunately, the Acme Cap is currently out of stock. You can try checking back later or purchasing a different item.",
  "sources": [],
  "confidence": 1.0,
  "answered": true
}
Tools called: ['buy_item']


---
# Task 4 · Prove It (Mini Eval)

> **Reference:** `04_prompt_optimization.ipynb` – eval table pattern

8 test cases: 4 handbook questions, 1 off-topic refusal, 1 greeting, 2 store actions.

### 4.1 — Define the test cases

In [20]:
CATALOG["sku-tee"]["stock"] = 10
CATALOG["sku-mug"]["stock"] = 5
CATALOG["sku-cap"]["stock"] = 0
ORDERS.clear()

TEST_CASES = [
    {
        "label": "Pro backup duration",
        "input": "What does the Pro plan include, how long are its backups retained, and what is its support response time?",
        "check": lambda r, t: (
            r.answered
            and "7" in r.answer
            and len(r.sources) > 0
            and "search_handbook" in t
        ),
    },
    {
        "label": "Refund window (annual)",
        "input": "What is the refund window for annual plans?",
        "check": lambda r, t: (
            r.answered
            and "30" in r.answer
            and len(r.sources) > 0
            and "search_handbook" in t
        ),
    },
    {
        "label": "Scale plan SLA",
        "input": "What is the Scale plan SLA?",
        "check": lambda r, t: (
            r.answered
            and ("4" in r.answer or "99.95" in r.answer)
            and len(r.sources) > 0
            and "search_handbook" in t
        ),
    },
    {
        "label": "Pro plan price",
        "input": "How much does the Pro plan cost?",
        "check": lambda r, t: (
            r.answered
            and "29" in r.answer
            and len(r.sources) > 0
            and "search_handbook" in t
        ),
    },
    {
        "label": "Off-topic (refuse)",
        "input": "What is the GDP of Saudi Arabia?",
        "check": lambda r, t: not r.answered,
    },
    {
        "label": "Greeting (no tool)",
        "input": "Hi there!",
        "check": lambda r, t: len(t) == 0,
    },
    {
        "label": "Buy 2 mugs (success)",
        "input": "I'd like to buy 2 mugs.",
        "check": lambda r, t: "buy_item" in t and r.answered,
    },
    {
        "label": "Buy cap (out of stock)",
        "input": "I want to buy the cap.",
        "check": lambda r, t: "buy_item" in t and any(
            w in r.answer.lower()
            for w in ["out of stock", "stock", "unavailable", "error", "not available", "cannot"]
        ),
    },
]
print(f"Defined {len(TEST_CASES)} test cases")

Defined 8 test cases


### 4.2 — Run the eval

In [21]:
results = []
for i, case in enumerate(TEST_CASES, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}/{len(TEST_CASES)}: {case['label']}")
    print(f"  Input: {case['input']}")
    try:
        reply, tools = grounded_assistant(case["input"], verbose=True)
        passed = case["check"](reply, tools)
        results.append({"case": case["label"], "pass": passed,
                        "tool": ", ".join(tools) if tools else "(none)",
                        "answered": reply.answered, "confidence": reply.confidence,
                        "answer_preview": reply.answer[:80]})
        print(f"  Result: [{'PASS' if passed else 'FAIL'}]")
        print(f"  Answer: {reply.answer[:100]}")
        print(f"  Sources: {reply.sources}")
        print(f"  Confidence: {reply.confidence}")
    except Exception as e:
        results.append({"case": case["label"], "pass": False, "tool": f"ERROR: {e}",
                        "answered": None, "confidence": None, "answer_preview": str(e)[:80]})
        print(f"  Result: [ERROR] — {e}")


Test 1/8: Pro backup duration
  Input: What does the Pro plan include, how long are its backups retained, and what is its support response time?
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] 1 tool call(s): search_handbook
    search_handbook({'query': 'What does the Pro plan include, how long are its backups retained, and what is its support response time?'}) -> {"context": "[1] ## Backups & Recovery\n- Pro: automatic daily backups retained for 7 days.\n- Scale: PITR with 1-second granularity, 35-day retention.\n- Restores can be triggered from the dashboard 
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Al

API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~1168 prompt tokens, ~29 completion tokens => $0.000193
  Result: [PASS]
  Answer: The Pro plan includes daily backups retained for 7 days, email support, and a first response within 
  Sources: ['Backups & Recovery', 'Support & SLA', 'Plans & Pricing']
  Confidence: 1.0

Test 2/8: Refund window (annual)
  Input: What is the refund window for annual plans?
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] 1 tool call(s): search_handbook
    search_handbook({'query': 'refund w

API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~916 prompt tokens, ~25 completion tokens => $0.000152
  Result: [FAIL]
  Answer: The GDP of Saudi Arabia is approximately $1.73 trillion (nominal) and $1.38 trillion (PPP) as of 202
  Sources: ['Economy of Saudi Arabia']
  Confidence: 0.9

Test 6/8: Greeting (no tool)
  Input: Hi there!
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] no tool calls -> structured parse


API call failed on attempt 1: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
Max retries exceeded. Total attempts: 1, Last error: Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}


  [Fallback Triggered] Primary failed: InstructorRetryException - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [Cost Estimate] ~910 prompt tokens, ~8 completion tokens => $0.000141
  Result: [PASS]
  Answer: Hello! How can I assist you today?
  Sources: []
  Confidence: 0.0

Test 7/8: Buy 2 mugs (success)
  Input: I'd like to buy 2 mugs.
  [Fallback Triggered] Primary failed: APIStatusError - Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
  [round 1] 1 tool call(s): buy_item
    buy_item({'item_id': 'sku-mug', 'quantity': 2}) -> {"order_id": "ord-54de70ea", "item": "Acme Mug", "quantity": 2, "total": 24.0, "status": "confirmed"}
  [Fallback Triggered] Primary fail

### 4.3 — Results table

In [22]:
print("\n" + "=" * 90)
print("EVAL RESULTS TABLE")
print("=" * 90)
print(f"{'Case':<30} {'Result':<8} {'Tool Called':<25} {'Answered':<10} {'Conf'}")
print("-" * 90)
total_pass = 0
for r in results:
    status   = "PASS" if r["pass"] else "FAIL"
    answered = str(r.get("answered", "?"))
    conf     = f"{r['confidence']:.1f}" if r["confidence"] is not None else "?"
    print(f"{r['case']:<30} {status:<8} {r['tool']:<25} {answered:<10} {conf}")
    if r["pass"]: total_pass += 1
print("-" * 90)
print(f"Total: {total_pass}/{len(results)} passed")
print("=" * 90)


EVAL RESULTS TABLE
Case                           Result   Tool Called               Answered   Conf
------------------------------------------------------------------------------------------
Pro backup duration            PASS     search_handbook           True       1.0
Refund window (annual)         PASS     search_handbook           True       0.9
Scale plan SLA                 PASS     search_handbook           True       1.0
Pro plan price                 PASS     search_handbook           True       1.0
Off-topic (refuse)             FAIL     (none)                    True       0.9
Greeting (no tool)             PASS     (none)                    False      0.0
Buy 2 mugs (success)           PASS     buy_item                  True       1.0
Buy cap (out of stock)         FAIL     (none)                    False      0.0
------------------------------------------------------------------------------------------
Total: 6/8 passed


### 4.4 — Bonus: test `return_item`

Test returning the order we just created and handling an unknown order gracefully.

In [23]:
if ORDERS:
    last_order_id = list(ORDERS.keys())[-1]
    print(f"Testing return of order: {last_order_id}")
    print("\n--- Return existing order ---")
    reply, tools = grounded_assistant(f"Please return order {last_order_id}")
    pretty(reply.model_dump(), title="Return Result")
    print(f"Tools: {tools}")
    print("\n--- Return unknown order ---")
    reply, tools = grounded_assistant("Please return order ord-doesnotexist")
    pretty(reply.model_dump(), title="Unknown Order Result")
    print(f"Tools: {tools}")
else:
    print("No orders to test with — run the buy test first.")

Testing return of order: ord-54de70ea

--- Return existing order ---
  [round 1] 1 tool call(s): return_item
    return_item({'order_id': 'ord-54de70ea'}) -> {"order_id": "ord-54de70ea", "status": "returned", "refunded": 24.0, "message": "Order 'ord-54de70ea' returned, $24.00 refunded."}
  [round 2] no tool calls -> structured parse
  [Cost Estimate] ~606 prompt tokens, ~16 completion tokens => $0.000101
Return Result
{
  "answer": "Order 'ord-54de70ea' has been returned. $24.00 has been refunded.",
  "sources": [],
  "confidence": 1.0,
  "answered": true
}
Tools: ['return_item']

--- Return unknown order ---
  [round 1] 1 tool call(s): return_item
    return_item({'order_id': 'ord-doesnotexist'}) -> {"error": "OrderNotFound", "message": "Order 'ord-doesnotexist' does not exist."}
  [round 2] no tool calls -> structured parse
  [Cost Estimate] ~595 prompt tokens, ~11 completion tokens => $0.000096
Unknown Order Result
{
  "answer": "Error: Order 'ord-doesnotexist' does not exist.",
  "

---
# Grounded vs. Ungrounded Comparison

Compare RAG-grounded answers vs. raw model to show why grounding matters.

In [26]:
comparison_questions = [
    "What is the refund policy for annual plans?",
    "What is the GDP of Saudi Arabia?",
]

for q in comparison_questions:
    print("=" * 70)
    print(f"Question: {q}")
    reply, _ = grounded_assistant(q, verbose=False)
    print(f"\n  GROUNDED (RAG):")
    print(f"    answered={reply.answered}, confidence={reply.confidence}")
    print(f"    answer: {reply.answer[:150]}")
    print(f"    sources: {reply.sources}")
    raw = _raw_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": q}],
    ).choices[0].message.content
    print(f"\n  UNGROUNDED (raw model):")
    print(f"    answer: {raw[:200]}")
    print()

Question: What is the refund policy for annual plans?

  GROUNDED (RAG):
    answered=True, confidence=0.95
    answer: Annual plans offer a prorated refund within the first 30 days of purchase; after that, no refunds are available.
    sources: ['Refund Policy']


NameError: name '_raw_client' is not defined

---
# REPORT

## Results Summary

The eval suite tests 8 cases covering:
- **4 handbook questions** — verified correct fact with cited source
- **1 off-topic question** — verified `answered=False` (refusal)
- **1 greeting** — verified no tool was called
- **2 store actions** — verified `buy_item` called, confirmed order/error handling

## How `instructor` Replaces `client.beta.chat.completions.parse`

| Step | OpenAI original | Groq + instructor |
|------|-----------------|-------------------|
| Patch client | *(built-in)* | `instructor.from_openai(client, mode=MD_JSON)` |
| Structured call | `client.beta.chat.completions.parse(response_format=GroundedReply)` | `client.chat.completions.create(response_model=GroundedReply)` |
| Get result | `.choices[0].message.parsed` | Return value directly |
| Works on Groq | No | Yes |

## API Summary

| Feature | Value |
|---------|-------|
| Chat model | `llama-3.1-8b-instant` via Groq |
| Embeddings | `nomic-embed-text-v1.5` via Groq |
| Structured output | `instructor` (MD_JSON mode) |
| Vector DB | ChromaDB in-memory, cosine similarity |
| Path setup | `pathlib.Path.cwd().parents[1] / "shared"` |

**Conclusion:** Grounding the assistant with RAG dramatically improves reliability.
The similarity threshold ensures the system refuses rather than guesses when it
does not have relevant context.

## Bonus: Streamlit Web UI 🌐
The cell below creates a Streamlit app (`streamlit_app.py`) that imports our `grounded_assistant` function and builds a simple web interface for it.

In [40]:
%%writefile streamlit_app.py
import streamlit as st
import time
from grounded_assistant import grounded_assistant

st.set_page_config(page_title="Acme Cloud Assistant", layout="centered")

st.title("☁️ Acme Cloud Support Assistant")
st.markdown("Ask any question about our policies, plans, or try buying/returning items!")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        # If assistant message has extra fields, display them
        if msg["role"] == "assistant" and "sources" in msg:
            if msg["sources"]:
                st.markdown("**📚 Sources:** " + ", ".join([f"`{s}`" for s in msg["sources"]]))
            if msg["tools"]:
                st.info(f"🔧 Tools used: {', '.join(msg['tools'])}")

# Generator for simulating a streaming effect
def stream_data(text):
    for word in text.split(" "):
        yield word + " "
        time.sleep(0.04)

# React to user input
if prompt := st.chat_input("Enter your question or request..."):
    # Display user message in chat message container
    st.chat_message("user").markdown(prompt)
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            reply, tools = grounded_assistant(prompt, verbose=False)
        
        # 1. Stream the answer
        response_text = reply.answer
        if not reply.answered:
            response_text = "⚠️ " + response_text
            
        st.write_stream(stream_data(response_text))
        
        # 2. Fill the rest of the fields after streaming
        if reply.sources:
            st.markdown("**📚 Sources:** " + ", ".join([f"`{s}`" for s in reply.sources]))
        else:
            st.caption("_No handbook sources used._")
            
        if tools:
            st.info(f"🔧 Tools called: {', '.join(tools)}")

    # Add assistant response to chat history
    st.session_state.messages.append({
        "role": "assistant", 
        "content": response_text,
        "sources": reply.sources,
        "tools": tools
    })

Overwriting streamlit_app.py


In [41]:
!python -m pip install streamlit -q
!python -m streamlit run streamlit_app.py

^C
